__*코드의 주요 특징*__

* 확률 기반 생성: 단순히 데이터를 채우는 것이 아니라, np.random.rand() < prob 방식을 사용하여 실제 환경처럼 불규칙하면서도 통계적 경향(여름철 집중 등)을 갖도록 했습니다.
* A현장 성별 로직: 남성 근로자가 더 많지만 사고 건수는 여성과 1:1이 되도록 0.5 확률로 성별을 할당했습니다.
* B현장 계절 비중: 요청하신 봄(30%), 여름(25%), 가을(20%), 겨울(25%)의 비율을 가중치로 환산하여 사고 발생 빈도를 조절했습니다.
* C현장 고령화: 평균 연령을 45세로 설정하여 다른 현장보다 높은 연령대의 데이터가 생성됩니다.

In [ ]:
### 데이터 생성
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. 설정 및 초기화
np.random.seed(42)
start_date = datetime(2021, 1, 1)
end_date = datetime(2023, 12, 31)
date_range = (end_date - start_date).days + 1

# 현장별 기본 정보 설정
sites_info = {
    'A': {'name': '아파트', 'workers': 300, 'avg_age': 40, 'edu': 30, 'female_ratio': 0.3},
    'B': {'name': '오피스텔', 'workers': 200, 'avg_age': 35, 'edu': 5, 'female_ratio': 0.0},
    'C': {'name': '일반사무용(30층)', 'workers': 250, 'avg_age': 45, 'edu': 15, 'female_ratio': 0.2}
}

accident_types = ['낙상', '화상', '단순 골절', '추락사']
p_types = [0.5, 0.2, 0.2, 0.1] # 낙상 비중을 높게 설정

accident_data = []

# 2. 데이터 생성 루프
for i in range(date_range):
    curr_date = start_date + timedelta(days=i)
    month = curr_date.month
    
    # 계절 판정
    if month in [3, 4, 5]: season = '봄'
    elif month in [6, 7, 8]: season = '여름'
    elif month in [9, 10, 11]: season = '가을'
    else: season = '겨울'
    
    for site_id, info in sites_info.items():
        # 현장별 기본 사고 확률 설정 (교육시간/인원 비례)
        base_prob = 0.005 
        
        if site_id in ['A', 'C']:
            # A, C현장: 여름에 사고 급증 (비 조건 반영)
            if season == '여름':
                prob = base_prob * 3.5
            else:
                prob = base_prob
        else: # B현장
            # 계절 비중 반영 (봄 30%, 여름 25%, 가을 20%, 겨울 25%)
            seasonal_weights = {'봄': 0.30, '여름': 0.25, '가을': 0.20, '겨울': 0.25}
            prob = base_prob * 2.0 * (seasonal_weights[season] / 0.25)

        # 사고 발생 여부 결정
        if np.random.rand() < prob:
            # 성별 결정 조건 반영
            if site_id == 'A':
                # A현장: 남자가 많으나 사고 건수는 남녀 동일 (5:5)
                gender = '여성' if np.random.rand() < 0.5 else '남성'
            elif site_id == 'B':
                # B현장: 남성으로만 구성
                gender = '남성'
            else: # C현장
                gender = '여성' if np.random.rand() < 0.5 else '남성'

            # 나이 생성 (평균 연령 기반 정규분포)
            age = int(np.random.normal(info['avg_age'], 5))
            
            # 데이터 저장
            accident_data.append({
                'Date': curr_date.strftime('%Y-%m-%d'),
                'Month': month,
                'Season': season,
                'Site': site_id,
                'Site_Name': info['name'],
                'Worker_Age': age,
                'Gender': gender,
                'Accident_Type': np.random.choice(accident_types, p=p_types),
                'Edu_Time': info['edu']
            })

# 3. 데이터프레임 변환 및 확인
df = pd.DataFrame(accident_data)

# 결과 출력
print(f"총 생성된 사고 건수: {len(df)}건")
print(df.head(10))

# CSV 파일로 저장하려면 아래 주석을 해제하세요.
df.to_csv('construction_accidents_3y_type1.csv', index=False, encoding='utf-8-sig')

: 

__*이 데이터가 알고리즘 테스트에 좋은 이유 (현실적 요소)*__

* 변수 간 상호작용 (Interaction Effects):
  * 단순히 '여름에 사고가 많다'가 아니라, '비가 오는(is_rainy) 날씨'와 '현장(A, C)'이 결합될 때만 확률이 튀도록 설계했습니다. 알고리즘이 이를 잡아내는지 테스트할 수 있습니다.
* 경험 수치 (Experience Years):
  * 나이와 별개로 '경력' 변수를 추가했습니다. 보통 숙련도가 낮을 때 사고가 많지만, 아주 높을 때(방심)도 사고가 나는 비선형적 패턴을 분석 모델이 학습하는지 확인할 수 있습니다.
* 데이터 불균형 (Imbalance):
  * 전체 날짜 대비 사고가 일어난 날은 매우 적습니다. 이는 실제 산재 데이터의 특징인 '희소성'을 반영하여, 알고리즘의 정밀도(Precision)와 재현율(Recall)을 제대로 평가할 수 있게 합니다.
* A현장의 역설 (Gender Paradox):
  * 근로자 수는 남자가 압도적이지만 사고 건수는 1:1인 상황을 만들었습니다. 이는 모델이 단순히 '남자가 많으니 남자가 위험하다'라고 판단하는 오류(Bias)를 범하는지 테스트하기 위함입니다.

이 데이터를 기반으로 XGBoost나 SHAP을 돌려보시면, 단순히 "A현장이 위험하다"가 아니라 "A현장은 비 오는 날 여성 근로자의 낙상 위험이 통계적으로 유의미하게 높다" 같은 구체적인 인사이트를 도출할 수 있습니다.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. 초기 설정
np.random.seed(77)
start_date = datetime(2021, 1, 1)
total_days = 365 * 3
accident_records = []

# 현장별 기본 속성
sites = {
    'A': {'name': '아파트', 'workers': 300, 'avg_age': 40, 'edu': 30, 'female_ratio': 0.3, 'safety_score': 0.8},
    'B': {'name': '오피스텔', 'workers': 200, 'avg_age': 35, 'edu': 5, 'female_ratio': 0.0, 'safety_score': 0.4},
    'C': {'name': '사무용빌딩', 'workers': 250, 'avg_age': 45, 'edu': 15, 'female_ratio': 0.2, 'safety_score': 0.6}
}

# 2. 날짜별 환경 데이터 및 사고 생성
for i in range(total_days):
    print(f"진행 중: {i+1}/{total_days}일", end='\r')
    curr_date = start_date + timedelta(days=i)
    month = curr_date.month
    day_of_week = curr_date.weekday() # 0:월, 6:일
    
    # [환경 변수] 계절 및 날씨 (여름철 강수 확률 증가)
    is_rainy = np.random.choice([True, False], p=[0.3, 0.7] if month in [6, 7, 8] else [0.1, 0.9])
    temp = np.random.normal(25, 5) if month in [6, 7, 8] else np.random.normal(5, 5)
    
    for sid, info in sites.items():
        # [사고 확률 계산 로직 - 현실성 강화]
        # 기본 확률은 현장의 안전점수(safety_score)에 반비례
        prob = 0.002 * (1 - info['safety_score'])
        
        # 조건 1: A, C현장은 비가 오면 위험도 4배 증가
        if sid in ['A', 'C'] and is_rainy:
            prob *= 4.0
            
        # 조건 2: B현장은 계절별 가중치 적용
        if sid == 'B':
            seasonal_map = {3: 1.5, 4: 1.5, 5: 1.5, 6: 1.2, 12: 1.2, 1: 1.2} # 봄철(3~5월) 가중치
            prob *= seasonal_map.get(month, 1.0)
            
        # 조건 3: 주말 전후(월, 금) 피로도에 따른 사고율 증가
        if day_of_week in [0, 4]:
            prob *= 1.3

        # 사고 발생 시뮬레이션
        if np.random.rand() < prob:
            # [개별 작업자 속성 생성]
            # A현장 성별 특이점: 남자가 많지만 사고 건수는 동일 (샘플링 편향 구현)
            if sid == 'A':
                gender = '여성' if np.random.rand() < 0.5 else '남성'
            else:
                gender = '여성' if np.random.rand() < info['female_ratio'] else '남성'
            
            age = int(np.random.normal(info['avg_age'], 7))
            experience = np.random.randint(1, 20) # 경력(년)
            
            # 사고 심각도 및 유형 (추락사는 고층인 C현장에서 더 높은 확률)
            acc_types = ['낙상', '화상', '단순 골절', '추락사']
            p_dist = [0.4, 0.2, 0.3, 0.1] if sid != 'C' else [0.3, 0.1, 0.3, 0.3]
            
            accident_records.append({
                'date': curr_date,
                'site_id': sid,
                'site_name': info['name'],
                'age': age,
                'gender': gender,
                'experience_years': experience,
                'is_rainy': is_rainy,
                'temperature': round(temp, 1),
                'edu_time': info['edu'],
                'accident_type': np.random.choice(acc_types, p=p_dist),
                'working_shift': np.random.choice(['주간', '야간'], p=[0.8, 0.2]),
                'overtime_hours': np.random.randint(0, 4)
            })

df_final = pd.DataFrame(accident_records)
print(f"생성 완료: 총 {len(df_final)}건의 사고 데이터")
df_final.to_csv('construction_accidents_3y_type2.csv', index=False, encoding='utf-8-sig')

생성 완료: 총 5건의 사고 데이터


__*이 데이터셋의 알고리즘 테스트 포인트 (Reality Check)*__

* 비상시 근로자 리스크:
  * 알고리즘이 '상시'보다 '비상시' 근로자의 사고 빈도가 인원수 대비 높다는 것을 찾아낼 수 있는지 확인합니다.
* 건물 규모와 사고 종류의 상관관계:
  * 70층인 C현장에서 '추락사' 비중이 월등히 높게 나타납니다. 층수 데이터를 사고 심각도와 연결하는 로직을 테스트하기 좋습니다.
* 야간 업무 제한:
  * B현장은 야간 업무가 없습니다. 알고리즘이 '근무 시간대'를 사고 예견의 중요한 변수로 사용하는지 볼 수 있습니다.
* 현장 규모 역설 (Simpson's Paradox):
  * 가장 크고 위험해 보이는 70층 건물(C)보다, 17층 아파트 10개 동을 짖는 A현장의 사고가 더 많습니다. 이는 "분산된 작업장(10개 동) 관리의 어려움"이라는 현실적인 시나리오를 반영한 것입니다.
* A현장 성별 편향 제거:
  * 남성이 훨씬 많음에도 사고 건수를 1:1로 맞췄기 때문에, 단순한 머신러닝은 "여성이 더 위험하다"고 오판할 수 있습니다. 이를 어떻게 보정하는지가 기술적 포인트입니다.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. 초기 설정 및 현장별 상세 스펙 반영
np.random.seed(42)
start_date = datetime(2021, 1, 1)
total_days = 365 * 3
accident_records = []

sites_config = {
    'A': {
        'name': '아파트(10개동/17층)', 
        'perm_workers': 300, 'temp_workers': 200, 
        'avg_age': 40, 'edu': 30, 'night_shift': True,
        'base_prob': 0.025  # 가장 높은 사고 빈도
    },
    'B': {
        'name': '오피스텔(2개동/20층)', 
        'perm_workers': 200, 'temp_workers': 0, 
        'avg_age': 35, 'edu': 5, 'night_shift': False,
        'base_prob': 0.018  # 중간 사고 빈도
    },
    'C': {
        'name': '초고층사무용(2개동/70층)', 
        'perm_workers': 250, 'temp_workers': 100, 
        'avg_age': 45, 'edu': 15, 'night_shift': True,
        'base_prob': 0.012  # 가장 낮은 사고 빈도 (안전관리 엄격 가정)
    }
}

# 2. 데이터 생성 루프
for i in range(total_days):
    curr_date = start_date + timedelta(days=i)
    month = curr_date.month
    season = '겨울'
    if 3 <= month <= 5: season = '봄'
    elif 6 <= month <= 8: season = '여름'
    elif 9 <= month <= 11: season = '가을'
    
    # 기상 조건 (여름철 비 확률)
    is_rainy = np.random.choice([True, False], p=[0.3, 0.7] if season == '여름' else [0.1, 0.9])

    for sid, info in sites_config.items():
        print(f"진행 중: {curr_date.strftime('%Y-%m-%d')} - 현장 {sid}", end='\r')
        # [확률 설계]
        prob = info['base_prob']
        
        # 조건: A, C는 여름(비)에 위험도 상승
        if sid in ['A', 'C'] and season == '여름' and is_rainy:
            prob *= 2.5
        
        # 조건: B는 요청하신 계절 비중 적용 (봄 30% 등)
        if sid == 'B':
            b_season_weight = {'봄': 1.5, '여름': 1.25, '가을': 1.0, '겨울': 1.25}
            prob *= b_season_weight[season]

        # 사고 발생 시뮬레이션
        if np.random.rand() < prob:
            # 근로자 유형 결정 (비상시 근로자가 있는 현장만)
            worker_type = '상시'
            if info['temp_workers'] > 0:
                total_w = info['perm_workers'] + info['temp_workers']
                # 비상시(임시) 근로자가 숙련도 부족으로 사고 확률이 1.5배 높다고 가정
                temp_p = (info['temp_workers'] * 1.5) / total_w
                worker_type = '비상시' if np.random.rand() < temp_p else '상시'

            # 성별 결정 (A현장: 남녀 사고 건수 동일 로직)
            if sid == 'A':
                gender = '여성' if np.random.rand() < 0.5 else '남성'
            elif sid == 'B':
                gender = '남성'
            else:
                gender = '여성' if np.random.rand() < 0.2 else '남성'

            # 사고 종류 (C현장은 70층이므로 추락사 비중 증가)
            acc_types = ['낙상', '화상', '단순 골절', '추락사']
            p_dist = [0.5, 0.2, 0.2, 0.1] if sid != 'C' else [0.3, 0.1, 0.2, 0.4]

            accident_records.append({
                'date': curr_date.strftime('%Y-%m-%d'),
                'site': sid,
                'site_name': info['name'],
                'worker_type': worker_type,
                'gender': gender,
                'age': int(np.random.normal(info['avg_age'], 6)),
                'season': season,
                'is_rainy': is_rainy,
                'working_shift': '주간' if not info['night_shift'] else np.random.choice(['주간', '야간'], p=[0.7, 0.3]),
                'accident_type': np.random.choice(acc_types, p=p_dist),
                'edu_time': info['edu']
            })

df_final = pd.DataFrame(accident_records)
print(f"총 {len(df_final)}건의 데이터 생성 완료.")
print(df_final.sample(10)) # 무작위 10개 확인
df_final.to_csv('construction_accidents_3y_type3.csv', index=False, encoding='utf-8-sig')